# Assigment 7 - Issue [#979](https://github.com/alexanderquispe/Diplomado_PUCP/issues/979)
**Deadline: September 7th - 23:59**

Fernando Mendoza | Andrea Pezo | Michel Cotrina | Estefanny Gil | Armando Ore

## Instructions

1. Import the data located at [this link](https://github.com/alexanderquispe/Diplomado_PUCP/blob/main/_data/data_dengue_peru.csv). It has information on people infected with dengue at the district level for 2015 to 2021.

2. Generate ubigeo for Departments and Provinces taking the first two and four numbers. Hint: [Use this code](https://stackoverflow.com/questions/35552874/get-first-letter-of-a-string-from-column).

3. Use geopandas to plot the number of cases in 2021 by the district using a continuous legend. Do not forget to indicate the color of NA values. Use this [shapefile](https://github.com/alexanderquispe/Diplomado_PUCP/tree/main/_data/LIMITE_DISTRITAL_2020_INEI).

4. Use geopandas to plot the number of cases in 2021 by the province using a continuous legend. Do not forget to indicate the color of NA values. Use this [shapefile](https://github.com/alexanderquispe/Diplomado_PUCP/tree/main/_data/LIMITE_DISTRITAL_2020_INEI). For this task, you will have to aggregate shapefiles at the province level.

5. Use geopandas to plot the number of cases by the department for all the years using subplots. --Every subplot for each year. Do not forget to indicate the color of NA values. Use this [shapefile](https://github.com/alexanderquispe/Diplomado_PUCP/tree/main/_data/LIMITE_DISTRITAL_2020_INEI). For this task, you will have to aggregate shapefiles at the department level.

6. Use geopandas to plot the number of cases by the department for all 2021 quarters using subplots. Every subplot for each quarter. Use a categorical legend with 5 bins. Do not forget to indicate the color of NA values. Use this [shapefile](https://github.com/alexanderquispe/Diplomado_PUCP/tree/main/_data/LIMITE_DISTRITAL_2020_INEI). For this task, you will have to aggregate shapefiles at the department level. Hint: Use Semana variable to group by quarters.

## Step 0: Libraries

In [ ]:
#!pip install geopandas
#!pip install mapclassify

In [ ]:
import matplotlib.pyplot as plt 
import mapclassify

import numpy as np
import pandas as pd
import geopandas as gpd

## Step 1: Load data

It has information on people infected with dengue at the district level for 2015 to 2021.

In [ ]:
pwd

In [ ]:
dengue = pd.read_csv( r"../../_data/data_dengue_peru.csv",
        dtype={'Ubigeo': 'str'}, 
        converters={'Casos': lambda x: float(x.replace(',', '')) if x != '' else np.nan})
dengue.head()

## Step 2: Generate ubigeo for Departments and Provinces taking the first two and four numbers

In [ ]:
# Generate department and province codes
dengue['Departamento_Ubigeo'] = dengue['Ubigeo'].str[:2]
dengue['Provincia_Ubigeo'] = dengue['Ubigeo'].str[:4]
dengue

## Step 3: Use geopandas to plot the number of cases in 2021 by the district using a continuous legend

In [ ]:
 # Filter dengue data in 2021
dengue_2021 = dengue[dengue['Año'] == 2021]

# Cases by district
cases_by_district = dengue_2021.groupby('Ubigeo')['Casos'].sum().reset_index()

# Load the shapefile
districts_gdf = gpd.read_file('../../_data/LIMITE_DISTRITAL_2020_INEI/INEI_LIMITE_DISTRITAL.shp')

# Merge cases data with districts geodataframe using Ubigeo column
merged_gdf = districts_gdf.merge(cases_by_district, left_on='CODIGO', right_on='Ubigeo', how='left')

# Plot set up
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Map plot setting specifying color for NaN values
merged_gdf.plot(column='Casos', cmap='YlOrRd', legend=True,
               missing_kwds={"color": "grey", "label": "No data"},
               ax=ax)

# Add a title and adjust the legend
ax.set_title('Casos de dengue por distrito el 2021', fontsize=15)
ax.set_axis_off()  # Turn off the axis

plt.show()

## Step 4: Use geopandas to plot the number of cases in 2021 by the province using a continuous legend

In [ ]:
# Load the dengue data file into the dengue DataFrame
dengue = pd.read_csv(r'../../_data/data_dengue_peru.csv',  
                     dtype={'Ubigeo': 'str'},  
                     converters={'Casos': lambda x: float(x.replace(',', '')) if x != '' else np.nan})  # Convert Casos to float

# Display the first few rows to confirm data is loaded correctly
display(dengue.head())


In [ ]:
# Filter data for 2021
dengue_2021 = dengue[dengue['Año'] == 2021]

# Aggregate cases by province using the "Provincia" column
cases_by_province = dengue_2021.groupby('Provincia')['Casos'].sum().reset_index()


In [ ]:
# Load the district-level shapefile (adjust the path as needed)
districts_gdf = gpd.read_file('../../_data/LIMITE_DISTRITAL_2020_INEI/INEI_LIMITE_DISTRITAL.shp')

# Display columns to identify relevant fields
print(districts_gdf.columns)


In [ ]:
# Ensure the "Ubigeo" column is read correctly and create "Provincia_Ubigeo"
districts_gdf['Provincia_Ubigeo'] = districts_gdf['UBIGEO'].str[:4] 


In [ ]:
# Dissolve district-level shapefile to create province-level shapefile based on "Provincia_Ubigeo"
provinces_gdf = districts_gdf.dissolve(by='Provincia_Ubigeo', aggfunc='sum').reset_index()


In [ ]:
# Check columns in dengue DataFrame
print(dengue.columns)


In [ ]:
# Ensure 'Ubigeo' is read correctly as a string and create 'Provincia_Ubigeo'
dengue['Provincia_Ubigeo'] = dengue['Ubigeo'].str[:4]  # First four digits for province

# Display the first few rows to confirm the column was created correctly
display(dengue.head())


In [ ]:
# Map 'Provincia_Ubigeo' codes back to 'Provincia' names using the dengue data
province_map = dengue[['Provincia', 'Provincia_Ubigeo']].drop_duplicates().set_index('Provincia_Ubigeo')['Provincia'].to_dict()

# Map the provinces to the shapefile DataFrame
provinces_gdf['Provincia'] = provinces_gdf['Provincia_Ubigeo'].map(province_map)


In [ ]:
# Merge cases data with the province shapefile using the "Provincia" column
merged_gdf_province = provinces_gdf.merge(cases_by_province, on='Provincia', how='left')

# Plot setup
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Map plot setting specifying color for NaN values
merged_gdf_province.plot(column='Casos', cmap='YlOrRd', legend=True,
                         missing_kwds={"color": "grey", "label": "No data"},
                         ax=ax)

# Add a title and adjust the legend
ax.set_title('Dengue Cases by Province in 2021', fontsize=15)
ax.set_axis_off()  # Turn off the axis

plt.show()


## Step 5: Use geopandas to plot the number of cases by the department for all the years using subplots.
Every subplot for each year. Do not forget to indicate the color of NA values. 

Use this shapefile. For this task, you will have to aggregate shapefiles at the department level.

In [ ]:
# Upload dengue data
dengue = pd.read_csv(r'../../_data/data_dengue_peru.csv',  
                     dtype={'Ubigeo': 'str'},  
                     converters={'Casos': lambda x: float(x.replace(',', '')) if x != '' else np.nan})

# Generate the department code from the Ubigeo (first 2 digits)
dengue['Departamento_Ubigeo'] = dengue['Ubigeo'].str[:2]


In [ ]:
# Load district shapefile (path provided)
districts_gdf = gpd.read_file('../../_data/LIMITE_DISTRITAL_2020_INEI/INEI_LIMITE_DISTRITAL.shp')

In [ ]:
# Create a list with the years to be graphed (2015-2021)
years = list(range(2015, 2022))

# Setting up the figure and subplots
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(20, 15))  # 3 rows and 3 columns for 7 years

# Flatten the subplot array for easy access
axes = axes.flatten()

# Iterate over each year and create a graph
for i, year in enumerate(years):
    # Filter dengue data for the current year
    dengue_year = dengue[dengue['Año'] == year]
    
    # Group cases by department
    cases_by_department = dengue_year.groupby('Departamento_Ubigeo')['Casos'].sum().reset_index()

    # Create department code from district shapefile (first 2 digits)
    districts_gdf['Departamento_Ubigeo'] = districts_gdf['UBIGEO'].str[:2]

    # Dissolve the districts to obtain geometry by department
    departments_gdf = districts_gdf.dissolve(by='Departamento_Ubigeo', aggfunc='sum').reset_index()

    # Merge case data with the departmental shapefile
    merged_gdf_department = departments_gdf.merge(cases_by_department, on='Departamento_Ubigeo', how='left')
    # Plot in the corresponding subplot
    ax = axes[i]
    merged_gdf_department.plot(column='Casos', cmap='YlOrRd', legend=True,
                               missing_kwds={"color": "lightgrey", "label": "No data"},
                               ax=ax, edgecolor="black", linewidth=0.5)
    
    # Adjust chart title
    ax.set_title(f'Dengue cases by department in {year}', fontsize=12)
    ax.set_axis_off()  # Turn off the axes

# Remove additional subplots if not used
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Adjust the layout so that the graphics do not overlap
plt.tight_layout()
plt.show()


## Step 6. Use geopandas to plot the number of cases by the department for all 2021 quarters using subplots.

In [ ]:
dengue_2021.head()

In [ ]:
# New column 'Quarter' based on the 'Semana' column
dengue_2021['Trimestre'] = pd.cut(dengue_2021['Semana'], bins=[0, 13, 26, 39, 53], labels=['Q1', 'Q2', 'Q3', 'Q4'])

# Check the first records to ensure that the quarters have been assigned correctly
print(dengue_2021[['Semana', 'Trimestre']].head())
dengue_2021.head()

In [ ]:
# Group cases by 'Departamento_Ubigeo' and 'Quarter'
dengue_departamento_trimestres = dengue_2021.groupby(['Departamento_Ubigeo', 'Trimestre'])['Casos'].sum().reset_index()

# Check the new DataFrame
dengue_departamento_trimestres.head()

In [ ]:
# Load the district shapefile (which will be aggregated at the departmental level)
districts_gdf = gpd.read_file('../../_data/LIMITE_DISTRITAL_2020_INEI/INEI_LIMITE_DISTRITAL.shp')

# Create the 'Departamento_Ubigeo' column in the shapefile
districts_gdf['Departamento_Ubigeo'] = districts_gdf['UBIGEO'].str[:2]

# Apply 'dissolve' to aggregate districts at the departmental level
departments_gdf = districts_gdf.dissolve(by='Departamento_Ubigeo', aggfunc='sum').reset_index()

# Merge the dissolved shapefile with the dengue data aggregated by quarter
merged_gdf_trimestres = departments_gdf.merge(dengue_departamento_trimestres, on='Departamento_Ubigeo', how='left')

# Check the merged data
merged_gdf_trimestres.head()

In [ ]:
# Set up the figure with 4 subplots, one for each quarter
fig, axes = plt.subplots(2, 2, figsize=(20, 15))  # 2 rows, 2 columns
quarters = ['Q1', 'Q2', 'Q3', 'Q4']

# Iterate over the quarters and create a map for each
for i, quarter in enumerate(quarters):
    ax = axes[i//2, i%2]  # Position in the correct subplot
    quarter_data = merged_gdf_trimestres[merged_gdf_trimestres['Trimestre'] == quarter]
    
    # Ensure missing values (NA) are correctly defined as NaN
    quarter_data['Casos'] = quarter_data['Casos'].fillna(np.nan)
    
    # Plot cases for this quarter using quantiles as the classification scheme
    quarter_data.plot(column='Casos', cmap='YlOrRd', legend=True, 
                      ax=ax, scheme='quantiles',edgecolor="black", linewidth=0.5,
                      missing_kwds={"color": "grey", "label": "No data"})
    
    # Add a title to the subplot
    ax.set_title(f'Dengue Cases in {quarter} of 2021', fontsize=14)
    ax.set_axis_off()  # Turn off the axes

# Adjust spacing between subplots
plt.tight_layout()
plt.show()